In [1]:
#
# How to use your trained model - Deep Learning basics with Python, TensorFlow and Keras
# Steg 5
#

In [2]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D
# more info on callbakcs: https://keras.io/callbacks/ model saver is cool too.
from tensorflow.keras.callbacks import TensorBoard
import pickle
import time


In [3]:
#
# Läser in tidigare sparat data från disk 

DATADIR = "C:\\LTH_data\\photo\\PythUtbPic\\"            # path till katalog med djurbilder
DATADIR_MOD = DATADIR + "pyth_mod_learn\\"               # path till sparad modell

TxT_INPmod_fileN = DATADIR_MOD + "X.pickle"              # Filnamn indata
TxT_OUTmod_fileN = DATADIR_MOD + "y.pickle"              # Filnamn utdata

print(f"  {TxT_INPmod_fileN  =  }")
print(f"  {TxT_OUTmod_fileN  =  }")

pickle_in = open(TxT_INPmod_fileN,"rb")                  # Läs Indata - bildmatrl
X = pickle.load(pickle_in)

pickle_in = open(TxT_OUTmod_fileN,"rb")                  # Läs Utdata - (0/1) (katt/hund)
y = pickle.load(pickle_in)

X = X/255.0                                              # Normera gråskallenivå, (0,255) transformeras till (0,1), utan medlevärdesförskjutning


  TxT_INPmod_fileN  =  'C:\\LTH_data\\photo\\PythUtbPic\\pyth_mod_learn\\X.pickle'
  TxT_OUTmod_fileN  =  'C:\\LTH_data\\photo\\PythUtbPic\\pyth_mod_learn\\y.pickle'


In [4]:
#
# Optimering map densitet, storlek och conv
#

#                        INKL. DROPOUT
dense_layers = [0]
layer_sizes = [64]
conv_layers = [2]
epochs_numb = 4
TxT_NAME = "theOPT_"              # Inledning på logg-fil-namnet
TxT_LOG_DIR = "logs_Opt"          # Path till logg-fil


TxT_MODEL_fName = DATADIR_MOD + '64x4-CNN.keras'    # Färdig anpassad modell (64 noder/lager & 4 iterationer)
print(f"  {TxT_MODEL_fName  =  }")



  TxT_MODEL_fName  =  'C:\\LTH_data\\photo\\PythUtbPic\\pyth_mod_learn\\64x4-CNN.keras'


In [5]:
#
# Nu körs Snurran endast för optimerings-värdena

snurra_number = 0

for dense_layer in dense_layers:
    for layer_size in layer_sizes:
        for conv_layer in conv_layers:
            print("="*40)
            snurra_number += 1
            print(f"   {snurra_number  =  }")
            named_tuple = time.localtime()                                # get struct_time
            time_string = time.strftime("%Y%m%d-%H%M", named_tuple)       # läsbart format
            TxT_LOGG_fNAME = "CNN-{}".format(time_string)
            TxT_LOGG_fNAME = "{}-conv-{}-nodes-{}-dense-{}".format(conv_layer, layer_size, dense_layer, TxT_LOGG_fNAME)
            TxT_LOGG_fNAME = "{}model-{}".format(TxT_NAME, TxT_LOGG_fNAME)
            print(f"  {TxT_LOGG_fNAME  =  }")

            model = Sequential()

            model.add(Conv2D(layer_size, (3, 3), input_shape=X.shape[1:]))
            model.add(Activation('relu'))
            model.add(MaxPooling2D(pool_size=(2, 2)))

            for l in range(conv_layer-1):
                model.add(Conv2D(layer_size, (3, 3)))
                model.add(Activation('relu'))
                model.add(MaxPooling2D(pool_size=(2, 2)))

            model.add(Flatten())

            for _ in range(dense_layer):
                model.add(Dense(layer_size))         # Lägg till ett Dence lager med "layer_size" antal neuroner, fullt klopplade Upp/Ned 
                model.add(Activation('relu'))        # Rectified Linear Unit
                model.add(Dropout(0.2))              # droppa 20% av neuronerna för att förhindra överträning (overfitting) 

            model.add(Dense(1))
            model.add(Activation('sigmoid'))

            #tensorboard = TensorBoard(log_dir="logs/{}".format(TxT_LOGG_fNAME))
            TxT_LOG_DIR_and_fNAME = "{}/{}".format(TxT_LOG_DIR, TxT_LOGG_fNAME)
            print(f"  {TxT_LOG_DIR_and_fNAME  =  }")
            tensorboard = TensorBoard(log_dir=TxT_LOG_DIR_and_fNAME)
            
            model.compile(loss='binary_crossentropy',
                          optimizer='adam',
                          metrics=['accuracy'],
                          )

            model.fit(X, y,
                      batch_size=32,
                      epochs=epochs_numb,
                      validation_split=0.3,
                      callbacks=[tensorboard])




   snurra_number  =  1
  TxT_LOGG_fNAME  =  'theOPT_model-2-conv-64-nodes-0-dense-CNN-20250502-0046'
  TxT_LOG_DIR_and_fNAME  =  'logs_Opt/theOPT_model-2-conv-64-nodes-0-dense-CNN-20250502-0046'


C:\Users\larth\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/4
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 175ms/step - accuracy: 0.5372 - loss: 0.7023 - val_accuracy: 0.5400 - val_loss: 0.6776
Epoch 2/4
22/22 ━━━━━━━━━━━━━━━━━━━━ 4s 163ms/step - accuracy: 0.6236 - loss: 0.6498 - val_accuracy: 0.6967 - val_loss: 0.6179
Epoch 3/4
22/22 ━━━━━━━━━━━━━━━━━━━━ 4s 176ms/step - accuracy: 0.7263 - loss: 0.5840 - val_accuracy: 0.6633 - val_loss: 0.6041
Epoch 4/4
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 210ms/step - accuracy: 0.7317 - loss: 0.5131 - val_accuracy: 0.7300 - val_loss: 0.5420


In [6]:
# Spara modellen som en '.keras'-fil för framtida behov

model.save(TxT_MODEL_fName)     # Sparar Modellen som ska användas för framtida TEST-prediktion
